# Main runs, type-1 methods, sweeps and label-free selection

**Computes:** FCM, the type-1 control, the Nie-Tan variant and the alternative reading of the interval algorithm on the main sets and the noise and imbalance sweeps, with label-free indices.

**Inputs:** UCI data, DIBCO images; the results of 05, if present, for a reproduction check.  
**Outputs:** `results/06/results_06.json`, checkpoints `runs06_*.pkl` (read by later notebooks) and a text report.


In [ ]:
import os, sys, time, json, pickle, platform, subprocess, glob, urllib.request
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DATASETS = None
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")

RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
OUTDIR = os.path.join(RESULTS_DIR, "06")
OUTDIR = os.path.abspath(OUTDIR)
for sub in ("data", "reports"):
    os.makedirs(os.path.join(OUTDIR, sub), exist_ok=True)
import numpy as np, scipy, sklearn
print(f"colab {IN_COLAB} | python {platform.python_version()} | numpy {np.__version__} | scipy {scipy.__version__} | sklearn {sklearn.__version__}")
print(f"output dir: {OUTDIR}   run stamp: {RUN_STAMP}")
J05_PATH = os.path.join(os.path.dirname(OUTDIR), "05", "results_05.json")

In [ ]:
MIRROR = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/"
FILES = ["sonar.csv", "pima-indians-diabetes.csv", "breast-cancer-wisconsin.data", "wheat-seeds.csv",
         "glass.csv", "ionosphere.csv", "banknote_authentication.csv", "ecoli.csv"]
DATA_DIR = os.path.join(OUTDIR, "data")
for f in FILES:
    p = os.path.join(DATA_DIR, f)
    if not os.path.exists(p):
        urllib.request.urlretrieve(MIRROR + f, p)
print("tabular files:", sorted(os.listdir(DATA_DIR)))
DIBCO_ROOT = os.environ.get("DIBCO_ROOT") or ("/content/dibco_all" if IN_COLAB else os.path.join(OUTDIR, "dibco_all"))
DIBCO_DIR = os.path.join(DIBCO_ROOT, "repo", "SauvolaNet", "Dataset")
if (DATASETS is None or "dibco" in DATASETS) and not os.path.isdir(DIBCO_DIR):
    print("cloning DIBCO (~945 MB) to", DIBCO_ROOT, "...")
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/RichSu95/Document_Binarization_Collection.git",
                    os.path.join(DIBCO_ROOT, "repo")], check=True)
if DATASETS is None or "dibco" in DATASETS:
    print(len(glob.glob(os.path.join(DIBCO_DIR, "*DIBCO*_source.*"))), "DIBCO images")

In [ ]:
import numpy as np
from itertools import permutations

EPS = 1e-12

def fcm_membership(D, m):
    p = 2.0 / (m - 1.0)
    z = -p * np.log(np.maximum(D, EPS))
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def hr_bounds(D, m1, m2):
    C = D.shape[1]
    U1, U2 = fcm_membership(D, m1), fcm_membership(D, m2)
    s = 1.0 / (D[:, :, None] / np.maximum(D[:, None, :], EPS)).sum(axis=2)
    cond = s < 1.0 / C
    upper = np.where(cond, U1, U2)
    lower = np.where(cond, U2, U1)
    return lower, upper

def km_interval_1d(x, wl, wu):
    o = np.argsort(x, kind="stable"); xs = x[o]; l = wl[o]; u = wu[o]
    n = len(x)
    cl, cu = np.cumsum(l), np.cumsum(u)
    clx, cux = np.cumsum(l * xs), np.cumsum(u * xs)
    Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
    k = np.arange(n + 1)
    Lk = np.concatenate([[0.0], cl]); Uk = np.concatenate([[0.0], cu])
    Lxk = np.concatenate([[0.0], clx]); Uxk = np.concatenate([[0.0], cux])
    vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
    vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
    kR, kL = int(np.argmax(vR_all)), int(np.argmin(vL_all))
    selR = np.zeros(n, bool); selL = np.zeros(n, bool)
    rank = np.empty(n, int); rank[o] = np.arange(n)
    selR[:] = rank >= kR
    selL[:] = rank < kL
    return float(vL_all[kL]), float(vR_all[kR]), selL, selR

def it2fcm_update(X, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    wl, wu = lower ** mc, upper ** mc
    for j in range(C):
        accL = np.zeros(N); accR = np.zeros(N)
        for l in range(M):
            vL, vR, selL, selR = km_interval_1d(X[:, l], wl[:, j], wu[:, j])
            V[j, l] = 0.5 * (vL + vR)
            accL += np.where(selL, upper[:, j], lower[:, j])
            accR += np.where(selR, upper[:, j], lower[:, j])
        uL[:, j], uR[:, j] = accL / M, accR / M
    return V, uL, uR

def dist(X, V):
    return np.maximum(np.linalg.norm(X[:, None, :] - V[None, :, :], axis=2), EPS)

def run_fcm(X, V0, m, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), m); W = U ** m
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    U = fcm_membership(dist(X, V), m)
    return V, U.argmax(1)

def run_it2fcm(X, V0, m1, m2, mc=None, iters=300, tol=1e-7):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update(X, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update(X, lo, up, mc)
    hard_hr = (0.5 * (uL + uR)).argmax(1)
    hard_nearest = dist(X, V).argmin(1)
    return V, hard_hr, hard_nearest

def accuracy(labels, pred, C):
    best = 0.0
    for perm in permutations(range(C)):
        best = max(best, float(np.mean(np.array(perm)[pred] == labels)))
    return best

def centre_error(V, centres):
    C = len(centres); best = np.inf
    for perm in permutations(range(C)):
        best = min(best, float(np.mean(np.linalg.norm(V[list(perm)] - centres, axis=1))))
    return best

def init_pp(X, C, rng):
    idx = [rng.integers(len(X))]
    for _ in range(1, C):
        d2 = np.min(((X[:, None, :] - X[idx][None]) ** 2).sum(2), axis=1)
        idx.append(rng.choice(len(X), p=d2 / d2.sum()))
    return X[idx].copy()

def scenario(name, seed):
    r = np.random.default_rng(seed)
    if name == "equal":
        c = np.array([[0.0, 0.0], [4.0, 0.0]]); n = (100, 100); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume":
        c = np.array([[0.0, 0.0], [5.0, 0.0]]); n = (100, 100); half = (2.5, 0.6)
        X = np.vstack([c[k] + r.uniform(-half[k], half[k], size=(n[k], 2)) for k in range(2)])
    elif name == "noise":
        c = np.array([[0.0, 0.0], [0.0, 4.0]]); n = (100, 100)
        X = np.vstack([r.normal(c[0], 1.0, size=(n[0], 2)), r.normal(c[1], 1.0, size=(n[1], 2)),
                       r.normal(c[0], 3.0, size=(40, 2))])
        y = np.r_[np.zeros(n[0]), np.ones(n[1]), np.zeros(40)].astype(int)
        return X, y, c, 2
    elif name == "density":
        c = np.array([[0.0, 0.0], [3.5, 0.0]]); n = (200, 30); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume3":
        c = np.array([[0.0, 0.0], [5.0, 0.0], [2.5, 4.0]]); n = (100, 100, 100); s = (1.4, 0.4, 0.8)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(3)])
    else:
        raise ValueError(name)
    y = np.concatenate([np.full(n[k], k) for k in range(len(n))])
    return X, y, c, len(n)

from scipy.optimize import linear_sum_assignment
def accuracy_h(labels, pred, C):
    M = np.zeros((C, C))
    np.add.at(M, (labels, pred), 1)
    r, c = linear_sum_assignment(-M)
    return float(M[r, c].sum() / len(labels))

def align(V, ref):
    Dm = np.linalg.norm(V[:, None] - ref[None], axis=2)
    r, c = linear_sum_assignment(Dm)
    out = np.zeros_like(ref); out[c] = V[r]
    return out

def centre_error_h(V, centres):
    return float(np.mean(np.linalg.norm(align(V, centres) - centres, axis=1)))

def real_dataset(name):
    from sklearn import datasets
    d = {"iris": datasets.load_iris, "wine": datasets.load_wine, "breast_cancer": datasets.load_breast_cancer}[name]()
    X = d.data.astype(float); X = (X - X.mean(0)) / X.std(0)
    y = d.target.astype(int)
    return X, y, len(np.unique(y))

def run_fcm_decoupled(X, V0, ma, mb, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), ma); W = U ** mb
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    return V, dist(X, V).argmin(1)

def it2fcm_update_fast(X, order, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    Xs = np.take_along_axis(X, order, axis=0)
    rank = np.empty_like(order); np.put_along_axis(rank, order, np.arange(N)[:, None].repeat(M, 1), axis=0)
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    z = np.zeros((1, M))
    for j in range(C):
        wl = (lower[:, j] ** mc)[order]; wu = (upper[:, j] ** mc)[order]
        cl, cu = np.cumsum(wl, 0), np.cumsum(wu, 0)
        clx, cux = np.cumsum(wl * Xs, 0), np.cumsum(wu * Xs, 0)
        Lk, Uk = np.vstack([z, cl]), np.vstack([z, cu]); Lxk, Uxk = np.vstack([z, clx]), np.vstack([z, cux])
        Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
        vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
        vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
        kR, kL = vR_all.argmax(0), vL_all.argmin(0)
        vR = vR_all[kR, np.arange(M)]; vL = vL_all[kL, np.arange(M)]
        V[j] = 0.5 * (vL + vR)
        selR = rank >= kR[None, :]; selL = rank < kL[None, :]
        up, lo = upper[:, j][:, None], lower[:, j][:, None]
        uR[:, j] = np.where(selR, up, lo).mean(1); uL[:, j] = np.where(selL, up, lo).mean(1)
    return V, uL, uR

def run_it2fcm_fast(X, V0, m1, m2, mc=None, iters=300, tol=1e-7, order=None):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    order = np.argsort(X, axis=0, kind="stable") if order is None else order
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1)

def real_dataset2(name, data_dir=DATA_DIR):
    import os, pandas as pd
    if name in ("iris", "wine", "wdbc"):
        return real_dataset({"wdbc": "breast_cancer"}.get(name, name))
    if name == "sonar":
        d = pd.read_csv(os.path.join(data_dir, "sonar.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = (d.iloc[:, -1] == "M").to_numpy(int)
    elif name == "pima":
        d = pd.read_csv(os.path.join(data_dir, "pima-indians-diabetes.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = d.iloc[:, -1].to_numpy(int)
    elif name == "wbc":
        d = pd.read_csv(os.path.join(data_dir, "breast-cancer-wisconsin.data"), header=None, na_values="?").dropna()
        X = d.iloc[:, 1:-1].to_numpy(float); y = (d.iloc[:, -1] == 4).to_numpy(int)
    else:
        raise ValueError(name)
    X = (X - X.mean(0)) / X.std(0)
    return X, y, len(np.unique(y))

EXTRA = {"seeds": "wheat-seeds.csv", "glass": "glass.csv", "ionosphere": "ionosphere.csv",
         "banknote": "banknote_authentication.csv", "ecoli": "ecoli.csv"}
def real_dataset3(name, data_dir=DATA_DIR):
    import os, pandas as pd
    if name not in EXTRA:
        return real_dataset2(name, data_dir)
    d = pd.read_csv(os.path.join(data_dir, EXTRA[name]), header=None)
    X = d.iloc[:, :-1].to_numpy(float)
    y = pd.factorize(d.iloc[:, -1].astype(str).str.strip(), sort=True)[0]
    X = X[:, X.std(0) > 0]
    X = (X - X.mean(0)) / X.std(0)
    return X, y.astype(int), len(np.unique(y))

In [ ]:
import time
import numpy as np

def run_it2fcm_nt(X, V0, m1, m2, mc=None, iters=300, tol=1e-6):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        W = 0.5 * (lo ** mc + up ** mc)
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    return V, (0.5 * (lo + up)).argmax(1)

def contingency(y, p, C):
    M = np.zeros((int(y.max()) + 1, C)); np.add.at(M, (y, p), 1); return M

def ari_from_cont(M):
    comb = lambda x: x * (x - 1) / 2.0
    n = M.sum(); s_ij = comb(M).sum(); a = comb(M.sum(1)).sum(); b = comb(M.sum(0)).sum()
    exp = a * b / comb(n); mx = 0.5 * (a + b)
    return float((s_ij - exp) / (mx - exp)) if mx != exp else 1.0

def nmi_from_cont(M):
    n = M.sum(); P = M / n; pi = P.sum(1); pj = P.sum(0)
    nz = P > 0
    I = float((P[nz] * np.log(P[nz] / (pi[:, None] * pj[None, :])[nz])).sum())
    H = lambda q: float(-(q[q > 0] * np.log(q[q > 0])).sum())
    d = 0.5 * (H(pi) + H(pj))
    return I / d if d > 0 else 1.0

def acc_from_cont(M):
    from scipy.optimize import linear_sum_assignment
    r, c = linear_sum_assignment(-M); return float(M[r, c].sum() / M.sum())

def scores(y, p, C):
    M = contingency(y, p, C)
    return dict(acc=acc_from_cont(M), ari=ari_from_cont(M), nmi=nmi_from_cont(M))

def silhouette_pre(Dsub, lab):
    ks = np.unique(lab)
    if len(ks) < 2: return -1.0
    n = len(lab); a = np.zeros(n); b = np.full(n, np.inf)
    for k in ks:
        mk = lab == k; cnt = mk.sum()
        s = Dsub[:, mk].sum(1)
        own = lab == k
        a[own] = s[own] / max(cnt - 1, 1)
        other = ~own
        b[other] = np.minimum(b[other], s[other] / cnt)
    sing = np.array([(lab == lab[i]).sum() == 1 for i in range(n)])
    sil = np.where(sing, 0.0, (b - a) / np.maximum(np.maximum(a, b), EPS))
    return float(sil.mean())

def calinski(X, lab):
    ks = np.unique(lab); n, k = len(X), len(ks)
    if k < 2 or k >= n: return -1.0
    mu = X.mean(0); B = W = 0.0
    for c in ks:
        Xc = X[lab == c]; mc_ = Xc.mean(0)
        B += len(Xc) * ((mc_ - mu) ** 2).sum(); W += ((Xc - mc_) ** 2).sum()
    return float((B / (k - 1)) / (W / (n - k))) if W > 0 else -1.0

def noise_instance(frac, seed):
    r = np.random.default_rng(seed); c = np.array([[0.0, 0.0], [0.0, 4.0]]); k = int(round(frac * 100))
    X = np.vstack([r.normal(c[0], 1.0, (100, 2)), r.normal(c[1], 1.0, (100, 2)), r.normal(c[0], 3.0, (k, 2))])
    y = np.r_[np.zeros(100), np.ones(100), np.zeros(k)].astype(int)
    return X, y, 2

def imbalance_instance(C, ratio, seed):
    r = np.random.default_rng(seed)
    R = 3.5 / (2 * np.sin(np.pi / C)); ang = 2 * np.pi * np.arange(C) / C
    c = np.c_[R * np.cos(ang), R * np.sin(ang)]
    n = np.maximum(np.round(100 * ratio ** (-np.arange(C) / max(C - 1, 1))).astype(int), 3)
    X = np.vstack([r.normal(c[k], 1.0, (n[k], 2)) for k in range(C)])
    y = np.concatenate([np.full(n[k], k) for k in range(C)])
    return X, y, C

In [ ]:
from PIL import Image
GRID = (1.05, 1.1, 1.3, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0)
PAIRS = [(a, b) for i, a in enumerate(GRID) for b in GRID[i + 1:]]
IT2_KEYS = [(a, b, mode) for (a, b) in PAIRS for mode in ("m1", "mid", "m2")]
DEC_KEYS = [(a, b) for a in GRID for b in GRID]
FAMILIES = {"FCM": list(GRID), "IT2FCM": IT2_KEYS, "IT2FCM-NT": IT2_KEYS, "T1dec": DEC_KEYS}
SYN = ("equal", "volume", "noise", "density", "volume3")
REAL = ("iris", "wine", "wdbc", "wbc", "sonar", "pima", "seeds", "glass", "ionosphere", "banknote", "ecoli")
NOISE = {f"noise{int(f*100):02d}": f for f in (0.0, 0.1, 0.2, 0.4, 0.6)}
IMB = {f"imb_C{C}_r{r}": (C, r) for C in (3, 5, 8) for r in (1, 5, 20)}
N_SYN, N_REAL, N_SWEEP, N_INIT, TOL, N_SIL = 40, 20, 20, 2, 1e-6, 1000

def get_instance(nm, s, data_dir):
    if nm in SYN:
        X, y, c, C = scenario(nm, 1000 + s); return X, y, C
    if nm in NOISE: return noise_instance(NOISE[nm], 6000 + s)
    if nm in IMB: return imbalance_instance(*IMB[nm], 7000 + s)
    X, y, C = real_dataset3(nm, data_dir); r = np.random.default_rng(3000 + s)
    idx = np.sort(np.concatenate([r.choice(np.where(y == k)[0], int(round(0.75 * (y == k).sum())), replace=False) for k in range(C)]))
    return X[idx], y[idx], C

def fit(fam, key, X, V0, order):
    if fam == "FCM": return run_fcm(X, V0, key, tol=TOL)
    if fam == "T1dec": return run_fcm_decoupled(X, V0, key[0], key[1], tol=TOL)
    a, b, mode = key; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
    if fam == "IT2FCM":
        V, hp, hn = run_it2fcm_fast(X, V0, a, b, mc=mc, tol=TOL, order=order); return V, hp
    return run_it2fcm_nt(X, V0, a, b, mc=mc, tol=TOL)

def run_instance06(X, y, C, s, evaluate=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    rec = {f: {k: dict(ari=[], acc=[], nmi=[], f=[], sil=[], ch=[]) for k in keys} for f, keys in FAMILIES.items()}
    secs = {f: 0.0 for f in FAMILIES}
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for fam, keys in FAMILIES.items():
            t0 = time.perf_counter()
            for k in keys:
                V, lab = fit(fam, k, X, V0, order)
                sc = evaluate(V, lab) if evaluate else scores(y, lab, C)
                d = rec[fam][k]
                for m in ("ari", "acc", "nmi"): d[m].append(sc[m])
                d["f"].append(sc.get("f", np.nan)); d["sil"].append(silhouette_pre(Dsub, lab[sub])); d["ch"].append(calinski(X, lab))
            secs[fam] += time.perf_counter() - t0
    return dict(rec=rec, sec_per_run={f: secs[f] / (N_INIT * len(FAMILIES[f])) for f in FAMILIES})

def dibco_instance(i, f, g):
    I8 = np.array(Image.open(f).convert("L")); G = np.array(Image.open(g).convert("L")) < 128
    hist_ink = np.bincount(I8[G].ravel(), minlength=256).astype(float)
    hist_bg = np.bincount(I8[~G].ravel(), minlength=256).astype(float)
    r = np.random.default_rng(5000 + i)
    X = (I8.ravel()[r.choice(I8.size, min(2000, I8.size), replace=False)] / 255.0)[:, None]
    def evaluate(V, lab):
        t = float(V.ravel().mean()); k = int(np.clip(np.ceil(t * 255.0), 0, 256))
        tp, fp = hist_ink[:k].sum(), hist_bg[:k].sum(); fn, tn = hist_ink.sum() - tp, hist_bg.sum() - fp
        M = np.array([[tn, fp], [fn, tp]])
        return dict(acc=float((tp + tn) / M.sum()), ari=ari_from_cont(M), nmi=nmi_from_cont(M),
                    f=float(2 * tp / max(2 * tp + fp + fn, 1)))
    out = run_instance06(X, np.zeros(len(X), int), 2, i, evaluate=evaluate)
    out["name"] = os.path.basename(f).replace("TRAIN_", "").replace("_source", "")
    return out

In [ ]:
from scipy import stats
MAIN = SYN + REAL + ("dibco",)
SWEEPS = tuple(NOISE) + tuple(IMB)
PAIRS_CMP = (("IT2FCM", "FCM"), ("IT2FCM", "T1dec"), ("IT2FCM", "IT2FCM-NT"), ("IT2FCM-NT", "T1dec"))

def halves(nm, R):
    ids = sorted(R); h = len(ids) // 2
    if nm != "dibco": return ids[:h], ids[h:]
    rng = np.random.default_rng(7); years = {i: R[i]["name"][5:9] for i in ids}; fold = {}
    for yv in sorted(set(years.values())):
        ix = [i for i in ids if years[i] == yv]; ix = [ix[j] for j in rng.permutation(len(ix))]
        for j, i in enumerate(ix): fold[i] = j % 2
    return [i for i in ids if fold[i] == 0], [i for i in ids if fold[i] == 1]

def tp(d):
    d = np.asarray(d, float)
    if len(d) < 2 or np.allclose(d, 0): return dict(mean=float(d.mean()) if len(d) else 0.0, t_p=1.0, w_p=1.0, n=len(d))
    t = float(stats.ttest_1samp(d, 0).pvalue)
    try: w = float(stats.wilcoxon(d).pvalue)
    except ValueError: w = 1.0
    return dict(mean=float(d.mean()), t_p=t if np.isfinite(t) else 1.0, w_p=w if np.isfinite(w) else 1.0, n=len(d))

def verdict(a, b):
    sig = lambda z: z["t_p"] < 0.05 and z["w_p"] < 0.05
    if sig(a) and sig(b) and a["mean"] > 0 and b["mean"] > 0: return "BETTER"
    if sig(a) and sig(b) and a["mean"] < 0 and b["mean"] < 0: return "WORSE"
    return "n.s."

def unsup_value(rec_f, metric, by):
    keys = list(rec_f); nq = len(rec_f[keys[0]][by]); vals = []
    for q in range(nq):
        k = max(keys, key=lambda kk: rec_f[kk][by][q])
        vals.append(rec_f[k][metric][q])
    return float(np.mean(vals))

def analyse_dataset(nm, R, metric):
    A, B = halves(nm, R)
    out = {"n": len(R)}
    for by in ("sil", "ch"):
        val = {f: {i: unsup_value(R[i]["rec"][f], metric, by) for i in R} for f in FAMILIES}
        cmp = {}
        for (x, z) in PAIRS_CMP:
            a = tp([val[x][i] - val[z][i] for i in A]); b = tp([val[x][i] - val[z][i] for i in B])
            cmp[f"{x} vs {z}"] = dict(A=a, B=b, verdict=verdict(a, b), mean=float(np.mean([val[x][i] - val[z][i] for i in R])))
        out[by] = dict(mean={f: float(np.mean(list(val[f].values()))) for f in FAMILIES}, cmp=cmp)
    lt = {}
    for sn, (tr, te) in (("A->B", (A, B)), ("B->A", (B, A))):
        sel = {f: max(FAMILIES[f], key=lambda k: np.mean([np.mean(R[i]["rec"][f][k][metric]) for i in tr])) for f in FAMILIES}
        test = {f: {i: float(np.mean(R[i]["rec"][f][sel[f]][metric])) for i in te} for f in FAMILIES}
        lt[sn] = dict(selected={f: str(sel[f]) for f in sel},
                      cmp={f"{x} vs {z}": tp([test[x][i] - test[z][i] for i in te]) for (x, z) in PAIRS_CMP})
    out["label_tuned"] = dict(splits=lt, verdict={c: verdict(lt["A->B"]["cmp"][c], lt["B->A"]["cmp"][c]) for c in lt["A->B"]["cmp"]})
    out["sec_per_run"] = {f: float(np.median([R[i]["sec_per_run"][f] for i in R])) for f in FAMILIES}
    return out

def dataset_level(RES, names, by, metric_label):
    rows = [n for n in names if n in RES]
    res = {}
    for (x, z) in PAIRS_CMP:
        d = np.array([RES[n][by]["cmp"][f"{x} vs {z}"]["mean"] for n in rows])
        w = float(stats.wilcoxon(d).pvalue) if np.any(np.abs(d) > 1e-12) else 1.0
        res[f"{x} vs {z}"] = dict(n=len(rows), median=float(np.median(d)), pos=int((d > 0).sum()), neg=int((d < 0).sum()), wilcoxon_p=w)
    M = np.array([[RES[n][by]["mean"][f] for f in FAMILIES] for n in rows])
    fr = stats.friedmanchisquare(*M.T)
    ranks = np.mean([stats.rankdata(-r) for r in M], axis=0)
    ps = []
    for f in FAMILIES:
        if f == "IT2FCM": continue
        d = M[:, list(FAMILIES).index("IT2FCM")] - M[:, list(FAMILIES).index(f)]
        ps.append((f, float(stats.wilcoxon(d).pvalue) if np.any(np.abs(d) > 1e-12) else 1.0))
    ps.sort(key=lambda t: t[1]); holm = {}; running = 0.0
    for j, (f, p) in enumerate(ps):
        running = max(running, min(1.0, (len(ps) - j) * p)); holm[f] = running
    res["friedman"] = dict(stat=float(fr.statistic), p=float(fr.pvalue), mean_rank=dict(zip(FAMILIES, map(float, ranks))), holm_vs_IT2FCM=holm)
    return res

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score, calinski_harabasz_score
X, y, C = real_dataset3("iris", DATA_DIR); V0 = init_pp(X, C, np.random.default_rng(0)); V, p = run_fcm(X, V0, 2.0)
s = scores(y, p, C); D = np.linalg.norm(X[:, None] - X[None], axis=2)
checks = {"ARI": abs(s["ari"] - adjusted_rand_score(y, p)), "NMI": abs(s["nmi"] - normalized_mutual_info_score(y, p)),
          "silhouette": abs(silhouette_pre(D, p) - silhouette_score(X, p)), "Calinski-Harabasz": abs(calinski(X, p) - calinski_harabasz_score(X, p)) / calinski(X, p)}
V2, p2 = run_it2fcm_nt(X, V0, 2.0, 2.0, mc=2.0); Vf, pf = run_fcm(X, V0, 2.0, tol=1e-6)
checks["IT2FCM-NT with m1 = m2 equals FCM"] = float(np.abs(V2 - Vf).max())
Xs, ys, cs, Cs = scenario("volume", 1); V0s = init_pp(Xs, Cs, np.random.default_rng(0))
checks["vectorized KM equals reference"] = float(np.abs(run_it2fcm(Xs, V0s, 1.5, 5.0)[0] - run_it2fcm_fast(Xs, V0s, 1.5, 5.0)[0]).max())
for k, v in checks.items(): print(f"{k:38s} {v:.2e}")
assert all(v < 1e-9 for v in checks.values())

In [ ]:
ALL = MAIN + SWEEPS if DATASETS is None else tuple(DATASETS)
def run_dataset(nm):
    path = os.path.join(OUTDIR, f"runs06_{nm}.pkl")
    out = pickle.load(open(path, "rb")) if os.path.exists(path) else {}
    t0 = time.time()
    if nm == "dibco":
        P = sorted((f, f.replace("_source.", "_target.")) for f in glob.glob(os.path.join(DIBCO_DIR, "*DIBCO*_source.*")))
        P = [(a, b) for a, b in P if os.path.exists(b)]
        todo = [i for i in range(len(P)) if i not in out]
        for i in todo:
            out[i] = dibco_instance(i, *P[i]); pickle.dump(out, open(path, "wb"))
    else:
        n = N_SYN if nm in SYN else (N_REAL if nm in REAL else N_SWEEP)
        todo = [s for s in range(n) if s not in out]
        for s in todo:
            X, y, C = get_instance(nm, s, DATA_DIR)
            out[s] = run_instance06(X, y, C, s); out[s]["C"] = C; pickle.dump(out, open(path, "wb"))
    print(f"{nm:14s} {len(out):3d} instances ({len(todo)} new) in {time.time()-t0:.0f}s", flush=True)
for nm in ALL:
    run_dataset(nm)

In [ ]:
RES = {}
for metric in ("ari", "acc", "nmi", "f"):
    RES[metric] = {}
    for nm in ALL:
        if metric == "f" and nm != "dibco": continue
        p = os.path.join(OUTDIR, f"runs06_{nm}.pkl")
        if os.path.exists(p): RES[metric][nm] = analyse_dataset(nm, pickle.load(open(p, "rb")), metric)
DL = {(by, metric): dataset_level(RES[metric], MAIN, by, metric) for by in ("sil", "ch") for metric in ("ari", "acc", "nmi")}
print("analyzed:", {m: len(v) for m, v in RES.items()})

In [ ]:
LINES = []
def say(s=""): print(s); LINES.append(s)
CMPS = [f"{x} vs {z}" for (x, z) in PAIRS_CMP]
def counts(R, names, by, c):
    v = [R[n][by]["cmp"][c]["verdict"] if by != "label_tuned" else R[n]["label_tuned"]["verdict"][c] for n in names if n in R]
    better = [n for n in names if n in R and (R[n][by]["cmp"][c]["verdict"] if by != "label_tuned" else R[n]["label_tuned"]["verdict"][c]) == "BETTER"]
    worse = [n for n in names if n in R and (R[n][by]["cmp"][c]["verdict"] if by != "label_tuned" else R[n]["label_tuned"]["verdict"][c]) == "WORSE"]
    return f"better {len(better):2d}  worse {len(worse):2d}  n.s. {len(v)-len(better)-len(worse):2d}   [+ {', '.join(better) or '-'} | - {', '.join(worse) or '-'}]"
say("=" * 88); say("06 -- label-free model selection, a second IT2 algorithm, noise and imbalance sweeps")
say(f"run {RUN_STAMP}"); say("=" * 88)
main = [n for n in MAIN if n in RES["ari"]]
say(f"main data sets: {len(main)}   sweep settings: {len([n for n in SWEEPS if n in RES['ari']])}"); say("")
for c in CMPS: say(f"  {c:24s}: {counts(RES['ari'], main, 'sil', c)}")
d = DL[("sil", "ari")]
for c in CMPS: say(f"  data-set level {c:24s}: median {d[c]['median']*100:+.2f} ARI pts, +{d[c]['pos']}/-{d[c]['neg']}, Wilcoxon p = {d[c]['wilcoxon_p']:.3g}")
fr = d["friedman"]
say(f"  Friedman over four methods: p = {fr['p']:.3g}; mean rank (1 = best) " + ", ".join(f"{f} {r:.2f}" for f, r in fr["mean_rank"].items()))
say("  Holm-corrected Wilcoxon, IT2FCM vs: " + ", ".join(f"{f} p = {p:.3g}" for f, p in fr["holm_vs_IT2FCM"].items()))
say("")
for (label, metric, by) in (("Calinski-Harabasz selection, ARI", "ari", "ch"), ("silhouette selection, accuracy", "acc", "sil"),
                            ("silhouette selection, NMI", "nmi", "sil"), ("label-tuned (05 protocol), ARI", "ari", "label_tuned"),
                            ("label-tuned (05 protocol), accuracy", "acc", "label_tuned")):
    say(f"  {label}")
    for c in CMPS[:2]: say(f"    {c:24s}: {counts(RES[metric], main, by, c)}")
say("")
if os.path.exists(J05_PATH):
    J = json.load(open(J05_PATH))["results"]; ok = tot = 0
    for n in main:
        m = "f" if n == "dibco" else "acc"
        if n not in J or n not in RES[m]: continue
        for c05, c06 in (("it2_vs_fcm", "IT2FCM vs FCM"), ("it2_vs_dec", "IT2FCM vs T1dec")):
            tot += 1; ok += J[n]["verdict"][c05] == RES[m][n]["label_tuned"]["verdict"][c06]
    say(f"REPRODUCTION OF 05 (label-tuned accuracy verdicts): {ok}/{tot} identical")
say("")
for n in SWEEPS:
    if n not in RES["ari"]: continue
    r = RES["ari"][n]["sil"]
    say(f"  {n:12s} " + " ".join(f"{f} {v:.3f}" for f, v in r["mean"].items()) +
        f" | vs FCM {r['cmp']['IT2FCM vs FCM']['verdict']:6s} vs T1dec {r['cmp']['IT2FCM vs T1dec']['verdict']}")
say("")
sec = {f: float(np.median([RES["ari"][n]["sec_per_run"][f] for n in main])) for f in FAMILIES}
say("RUN TIME (median over data sets of the median seconds per run): " + ", ".join(f"{f} {v*1000:.1f} ms" for f, v in sec.items())
    + f"  -> IT2FCM / FCM = {sec['IT2FCM']/sec['FCM']:.1f}x")
payload = dict(notebook="06", run_stamp=RUN_STAMP,
               results=RES, dataset_level={f"{k[0]}:{k[1]}": v for k, v in DL.items()})
report = "\n".join(LINES) + "\n"
paths = [os.path.join(OUTDIR, "results_06.json"), os.path.join(OUTDIR, "results_06_report.txt"),
         os.path.join(OUTDIR, "reports", f"results_06_{RUN_STAMP}.json"),
         os.path.join(OUTDIR, "reports", f"results_06_report_{RUN_STAMP}.txt")]
for p in paths:
    with open(p, "w") as fh:
        if p.endswith(".json"): json.dump(payload, fh, indent=1, default=str)
        else: fh.write(report)
        fh.flush(); os.fsync(fh.fileno())
print()
for p in paths:
    ok_ = os.path.exists(p) and os.path.getsize(p) > 0
    print(f"{'written' if ok_ else 'NOT WRITTEN'}: {p} ({os.path.getsize(p) if ok_ else 0} bytes)")